# Lab 5 — Exploración de Imágenes Médicas con DICOM

**Curso:** BE3006 · Análisis de Datos Biomédicos · UVG
**Referencia:** Clase 10 — Más Allá de la Tabla: Datos No Estructurados en Salud

---

## ¿De qué se trata este lab?

En los labs anteriores trabajamos con datos **tabulares**: tablas de pacientes, diagnósticos codificados, signos vitales numéricos, modelos de regresión. Todo vivía en filas y columnas.

Pero en la práctica clínica, una enorme cantidad de datos son **no estructurados**. Las imágenes médicas son el ejemplo más importante: radiografías, tomografías (CT), resonancias magnéticas (MR), ultrasonidos.

El estándar **DICOM** (Digital Imaging and Communications in Medicine) es el formato universal para imágenes médicas. Un archivo DICOM no es simplemente una imagen — es un **contenedor** que empaqueta:

- La **imagen** como una matriz de píxeles
- **Metadata clínica**: datos del paciente, del estudio, del equipo
- **Parámetros de visualización**: cómo interpretar y mostrar los píxeles

### Objetivos de aprendizaje

1. Entender la estructura de un archivo DICOM y su metadata clínica
2. Comparar metadata entre diferentes modalidades de imagen (CT, MR, radiografía)
3. Convertir valores de píxel a unidades Hounsfield (HU) y aplicar **windowing**
4. Construir un volumen 3D a partir de múltiples cortes DICOM
5. Segmentar tejidos usando umbrales de intensidad

### Estructura del lab

| Parte | Tema | Duración estimada |
|-------|------|-------------------|
| 0 | Setup y carga de datos | 10 min |
| 1 | Estructura y metadata DICOM | 45-60 min |
| 2 | Visualización: windowing y volumen 3D | 60-75 min |
| 3 | Segmentación por umbral | 30-45 min |

---

## Parte 0: Setup

### ¿Qué librerías usamos?

- **pydicom**: librería especializada para leer archivos DICOM. Es liviana (~2 MB) y no requiere dependencias pesadas.
- **matplotlib**: para visualizar las imágenes
- **numpy**: para manipular las matrices de píxeles
- **pandas**: para tablas comparativas de metadata

In [1]:
import pydicom
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print(f"pydicom version: {pydicom.__version__}")
print("Librerías cargadas correctamente.")

pydicom version: 3.0.2
Librerías cargadas correctamente.


### Carga de archivos de prueba

`pydicom` incluye archivos DICOM de prueba que podemos usar sin descargar nada. Vamos a cargar archivos de tres modalidades diferentes:

- **CT** (Tomografía Computarizada): usa rayos X para crear imágenes transversales del cuerpo
- **MR** (Resonancia Magnética): usa campos magnéticos, no hay radiación ionizante
- **CR** (Radiografía Computarizada): la "placa" digital tradicional

Los archivos se acceden con `pydicom.data.get_testdata_file(nombre)`, que retorna la ruta al archivo incluido en la instalación de pydicom.

In [2]:
# ── Cargar archivos de prueba ──
# get_testdata_file() retorna la ruta completa al archivo incluido en pydicom
ct_path = pydicom.data.get_testdata_file("CT_small.dcm")
mr_path = pydicom.data.get_testdata_file("MR_small.dcm")
mr_big_path = pydicom.data.get_testdata_file("MR-SIEMENS-DICOM-WithOverlays.dcm")
rg_path = pydicom.data.get_testdata_file("RG1_UNCI.dcm")

# Verificamos que los archivos existen
for name, path in [("CT_small", ct_path), ("MR_small", mr_path),
                   ("MR-SIEMENS", mr_big_path), ("RG1", rg_path)]:
    size_kb = os.path.getsize(path) / 1024
    print(f"{name:15s} {size_kb:8.1f} KB  {path}")

# También localizamos la serie CT con múltiples cortes (para el volumen 3D)
dicomdir_base = os.path.join(os.path.dirname(pydicom.data.__file__),
                              'test_files', 'dicomdirtests')
ct_series_path = os.path.join(dicomdir_base, '98892001', 'CT5N')
ct_series_files = sorted([os.path.join(ct_series_path, f)
                          for f in os.listdir(ct_series_path)])
print(f"\nSerie CT5N:     {len(ct_series_files)} cortes en {ct_series_path}")

MR-SIEMENS-DICOM-WithOverlays.dcm:   0%|          | 125/511k [00:00<13:57, 610B/s] 
RG1_UNCI.dcm:   0%|          | 1.76k/7.20M [00:00<1:03:36, 1.89kB/s]

CT_small            38.3 KB  /opt/conda/lib/python3.11/site-packages/pydicom/data/test_files/CT_small.dcm
MR_small             9.6 KB  /opt/conda/lib/python3.11/site-packages/pydicom/data/test_files/MR_small.dcm
MR-SIEMENS         499.0 KB  /home/jovyan/.pydicom/data/MR-SIEMENS-DICOM-WithOverlays.dcm
RG1               7031.6 KB  /home/jovyan/.pydicom/data/RG1_UNCI.dcm

Serie CT5N:     5 cortes en /opt/conda/lib/python3.11/site-packages/pydicom/data/test_files/dicomdirtests/98892001/CT5N


---

## Parte 1: Estructura y Metadata DICOM

### 1.1 ¿Qué hay dentro de un archivo DICOM?

Cuando cargamos un archivo DICOM con `pydicom.dcmread()`, obtenemos un **dataset** — un objeto que contiene todos los campos del archivo. Cada campo tiene:

- Un **tag** (identificador numérico, ej: `(0010, 0010)`)
- Un **nombre** legible (ej: `PatientName`)
- Un **valor** (ej: `"García^María"`)

Vamos a cargar el archivo CT y explorar su contenido completo. Observa la **cantidad** de información disponible — mucho más que una simple imagen.

In [3]:
# Cargamos el archivo CT
ct = pydicom.dcmread(ct_path)

# Imprimimos TODO el dataset — observa cuántos campos hay
print(f"Tipo de objeto: {type(ct)}")
print(f"Número de campos: {len(ct)}")
print()
print("=" * 70)
print("CONTENIDO COMPLETO DEL ARCHIVO DICOM (CT)")
print("=" * 70)
print(ct)

Tipo de objeto: <class 'pydicom.dataset.FileDataset'>
Número de campos: 258

CONTENIDO COMPLETO DEL ARCHIVO DICOM (CT)
Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 192
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.6.1.4.1.5962.1.1.1.1.1.20040119072730.12322
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.6.1.4.1.5962.2
(0002,0013) Implementation Version Name         SH: 'DCTOOL100'
(0002,0016) Source Application Entity Title     AE: 'CLUNIE1'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL']
(0008,0012) Instance Creation Date              DA: '20040119'
(000

### 1.2 Campos clave de un archivo DICOM

De todos esos campos, hay grupos que son especialmente importantes. Vamos a extraerlos uno por uno para entender qué información contienen.

Los campos se acceden como atributos del dataset (ej: `ds.PatientName`). Pero **no todos los campos existen en todos los archivos** — depende de la modalidad y del equipo. Por eso usamos `getattr(ds, campo, valor_por_defecto)` para manejar campos ausentes sin que el código falle.

In [4]:
# ── Datos del paciente ──
print("=== Datos del paciente ===")
print(f"  PatientID:        {ct.PatientID}")
print(f"  PatientName:      {ct.PatientName}")
print(f"  PatientBirthDate: {getattr(ct, 'PatientBirthDate', 'No disponible')}")
print(f"  PatientSex:       {ct.PatientSex}")

# ── Datos del estudio ──
print("\n=== Datos del estudio ===")
print(f"  StudyDate:        {ct.StudyDate}")
print(f"  StudyDescription: {getattr(ct, 'StudyDescription', 'No disponible')}")
print(f"  InstitutionName:  {ct.InstitutionName}")

# ── Datos de la imagen ──
print("\n=== Datos de la imagen ===")
print(f"  Modality:         {ct.Modality}")
print(f"  Rows x Columns:   {ct.Rows} x {ct.Columns}")
print(f"  PixelSpacing:     {ct.PixelSpacing} mm")
print(f"  SliceThickness:   {ct.SliceThickness} mm")
print(f"  BitsAllocated:    {ct.BitsAllocated} bits")

# ── Datos de visualización / calibración ──
print("\n=== Datos de calibración (para convertir a Hounsfield) ===")
print(f"  RescaleIntercept: {getattr(ct, 'RescaleIntercept', 'No disponible')}")
print(f"  RescaleSlope:     {getattr(ct, 'RescaleSlope', 'No disponible')}")
print(f"  WindowCenter:     {getattr(ct, 'WindowCenter', 'No disponible')}")
print(f"  WindowWidth:      {getattr(ct, 'WindowWidth', 'No disponible')}")

=== Datos del paciente ===
  PatientID:        1CT1
  PatientName:      CompressedSamples^CT1
  PatientBirthDate: 
  PatientSex:       O

=== Datos del estudio ===
  StudyDate:        20040119
  StudyDescription: e+1
  InstitutionName:  JFK IMAGING CENTER

=== Datos de la imagen ===
  Modality:         CT
  Rows x Columns:   128 x 128
  PixelSpacing:     [0.661468, 0.661468] mm
  SliceThickness:   5.000000 mm
  BitsAllocated:    16 bits

=== Datos de calibración (para convertir a Hounsfield) ===
  RescaleIntercept: -1024
  RescaleSlope:     1
  WindowCenter:     No disponible
  WindowWidth:      No disponible


### 1.3 Ejercicio: Función `dicom_summary()`

**Ejercicio 1.3:** Crea una función `dicom_summary(filepath)` que reciba la ruta de un archivo DICOM y retorne un diccionario con los campos clave. La función debe manejar campos ausentes con valores por defecto (`"N/A"`).

Campos a incluir: `PatientID`, `PatientName`, `Modality`, `Rows`, `Columns`, `PixelSpacing`, `SliceThickness`, `BitsAllocated`, `RescaleIntercept`, `RescaleSlope`, `WindowCenter`, `WindowWidth`, `InstitutionName`.

**Pista:** Usa `getattr(ds, campo, "N/A")` para cada campo.

In [7]:
def dicom_summary(filepath):
    """
    Carga un archivo DICOM y retorna un diccionario con los campos clave.
    Maneja campos ausentes con valor por defecto "N/A".
    """
    ds = pydicom.dcmread(filepath)

    # TU CÓDIGO AQUÍ: construye y retorna un diccionario con los campos listados arriba
    # Ejemplo para el primer campo:
    summary = {
        "PatientID": getattr(ds, "PatientID", "N/A"),
        "PatientName": getattr(ds, "PatientName", "N/A"),
        "Modality" : getattr(ds, "Modality", "N/A"),
        "Rows" : getattr(ds, "Rows", "N/A"),
        "Columns" : getattr(ds, "Columns", "N/A"),
        "PixelSpacing" : getattr(ds, "PixelSpacing", "N/A"),
        "SliceThickness" : getattr(ds, "SliceThickness", "N/A"),
        "BitsAllocated" : getattr(ds, "BitsAllocated", "N/A"),
        "RescaleIntercept" : getattr(ds, "RescaleIntercept", "N/A"),
        "RescaleSlope" : getattr(ds, "RescaleSlope", "N/A"),
        "WindowCenter" : getattr(ds, "WindowCenter", "N/A"),
        "WindowWidth" : getattr(ds, "WindowWidth", "N/A"),
        "InstitutionName" : getattr(ds, "InstitutionName", "N/A"),
                                  
    }
    return summary 

# Prueba tu función:
summary = dicom_summary(ct_path)

for key, value in summary.items():
    print(f"  {key:20s} {value}")
    

  PatientID            1CT1
  PatientName          CompressedSamples^CT1
  Modality             CT
  Rows                 128
  Columns              128
  PixelSpacing         [0.661468, 0.661468]
  SliceThickness       5.000000
  BitsAllocated        16
  RescaleIntercept     -1024
  RescaleSlope         1
  WindowCenter         N/A
  WindowWidth          N/A
  InstitutionName      JFK IMAGING CENTER


### 1.4 Comparar modalidades

DICOM es un estándar **común a múltiples modalidades** de imagen. Pero la metadata varía según el tipo de estudio. Por ejemplo:

- **CT** tiene `RescaleIntercept` y `RescaleSlope` (para convertir a unidades Hounsfield)
- **MR** típicamente no tiene Rescale, pero sí `WindowCenter` y `WindowWidth`
- **CR** (radiografía) tiene resolución muy alta pero sin información de profundidad

Vamos a comparar los cuatro archivos que cargamos.

In [8]:
# Cargamos los 4 archivos
datasets = {
    "CT (128x128)": pydicom.dcmread(ct_path),
    "MR (64x64)": pydicom.dcmread(mr_path),
    "MR-Siemens (484x484)": pydicom.dcmread(mr_big_path),
    "CR Radiografía (1955x1841)": pydicom.dcmread(rg_path),
}

# Campos a comparar
fields = ["Modality", "Rows", "Columns", "BitsAllocated", "PixelSpacing",
          "SliceThickness", "RescaleIntercept", "RescaleSlope",
          "WindowCenter", "WindowWidth"]

# Construimos tabla comparativa
rows = []
for field in fields:
    row = {"Campo": field}
    for name, ds in datasets.items():
        val = getattr(ds, field, "---")
        # Acortamos valores muy largos para la tabla
        val_str = str(val)
        if len(val_str) > 25:
            val_str = val_str[:22] + "..."
        row[name] = val_str
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("Campo")
print("=== Comparación de modalidades DICOM ===")
print()
comparison

=== Comparación de modalidades DICOM ===



,CT (128x128),MR (64x64),MR-Siemens (484x484),CR Radiografía (1955x1841)
Campo,,,,
Modality,CT,MR,MR,CR
Rows,128,64,484,1955
Columns,128,64,484,1841
BitsAllocated,16,16,16,16
PixelSpacing,"[0.661468, 0.661468]","[0.3125, 0.3125]","[0.72314049586777, 0.7...","[0.000, 0.000]"
SliceThickness,5.000000,0.8000,4,---
RescaleIntercept,-1024,---,---,---
RescaleSlope,1,---,---,---
WindowCenter,---,600,"[450, 200]",15000


### Preguntas de reflexión — Parte 1

Responde en las celdas de abajo:

**Pregunta 1.1:** ¿Qué información del archivo DICOM se perdería si la imagen se exportara como PNG o JPEG? Menciona al menos 3 campos específicos y explica por qué son importantes clínicamente.

Si el archivo DICOM se exportara como PNG se perderían los metadatos que es lo que caracteriza. Tres campos serían "Patient Id", "Pixel Spacing" y "Bits Allocated". Son importantes clínicamente porque dan información necesaria para poder hacer un diagnóstico. 

**Pregunta 1.2:** `PixelSpacing` indica la distancia física (en mm) entre píxeles, y `SliceThickness` el grosor del corte. ¿Por qué estos campos son indispensables para cualquier medición cuantitativa sobre la imagen (por ejemplo, medir el tamaño de un tumor)?

PixelSpacing: [0.661468, 0.661468] - 0.66 mm en x y en y 
SliceThickness: 5.00 mm en z 
Porque dan la información sobre la resolución respecto a los ejes y hacer la construcción en 3D son indispensables para mediciones cuantitativas porque vuelven la imagen un modelo matemático que permite que los doctores vean ciertos puntos en la imagen desde diferentes vistas y esto garaniza la precisión al realizar operaciones. 

**Pregunta 1.3:** `PatientName` y `PatientID` están embebidos dentro del archivo DICOM. ¿Qué implicaciones tiene esto para la privacidad del paciente? ¿Qué pasaría si compartes un archivo DICOM sin anonimizar?

Estos campos son datos sensibles del paciente por lo que para compartirlo se deben anonimizar. Si el archivo DICOM se comparte sin anonimizar los datos del paciente corren el riesgo de quedar expuestos y violar su privacidad. 

**CHECKPOINT 1** — Haz commit de tu progreso:
```bash
git add -A && git commit -m "Checkpoint 1: DICOM metadata exploration complete"
```

---

## Parte 2: Visualización e Interpretación

### 2.1 La imagen raw vs la imagen clínica

Empecemos por ver qué contiene la imagen CT "cruda" — los valores de píxel tal como están almacenados en el archivo.

In [ ]:
# Extraemos la matriz de píxeles del CT
pixel_array = ct.pixel_array

print(f"Tipo de datos:  {pixel_array.dtype}")
print(f"Forma (shape):  {pixel_array.shape}")
print(f"Valor mínimo:   {pixel_array.min()}")
print(f"Valor máximo:   {pixel_array.max()}")

# Visualizamos la imagen "cruda" (sin ningún procesamiento)
plt.figure(figsize=(6, 6))
plt.imshow(pixel_array, cmap='gray')
plt.colorbar(label='Valor de píxel (raw)')
plt.title('Imagen CT - valores crudos (sin procesar)')
plt.tight_layout()
plt.show()

### 2.2 Unidades Hounsfield (HU)

Los valores crudos de píxel no tienen significado clínico directo. Para interpretarlos necesitamos convertirlos a **unidades Hounsfield (HU)**, que representan la densidad del tejido:

| Tejido | HU aproximado |
|--------|---------------|
| Aire | -1000 |
| Pulmón | -500 |
| Grasa | -100 a -50 |
| Agua | 0 |
| Tejido blando | +20 a +80 |
| Hueso esponjoso | +300 a +500 |
| Hueso cortical | +500 a +2000 |

La conversión usa dos campos del DICOM:

```
HU = pixel_value × RescaleSlope + RescaleIntercept
```

En nuestro CT: `RescaleSlope = 1` y `RescaleIntercept = -1024`, entonces:

```
HU = pixel_value × 1 + (-1024) = pixel_value - 1024
```

In [ ]:
# Convertimos a unidades Hounsfield
slope = float(ct.RescaleSlope)
intercept = float(ct.RescaleIntercept)
image_hu = pixel_array.astype(np.float64) * slope + intercept

print(f"RescaleSlope:     {slope}")
print(f"RescaleIntercept: {intercept}")
print(f"HU mínimo:        {image_hu.min():.0f}")
print(f"HU máximo:        {image_hu.max():.0f}")

# Visualizamos en HU
plt.figure(figsize=(6, 6))
plt.imshow(image_hu, cmap='gray')
plt.colorbar(label='Unidades Hounsfield (HU)')
plt.title('Imagen CT en unidades Hounsfield')
plt.tight_layout()
plt.show()

print("\nLa imagen se ve igual, pero ahora los valores tienen significado clínico:")
print(f"  Un píxel con valor {image_hu.min():.0f} HU corresponde aproximadamente a aire")
print(f"  Un píxel con valor {image_hu.max():.0f} HU corresponde aproximadamente a hueso")

### 2.3 Windowing: la misma imagen, diferentes historias

Aquí viene el concepto más importante de este lab.

El rango de HU en una imagen CT puede ir de -1000 a +2000 (3000 valores). Pero un monitor solo muestra ~256 tonos de gris. Si mapeamos todo el rango a esos 256 tonos, cada tono cubre ~12 HU — demasiado grueso para distinguir tejidos que difieren en pocos HU.

El **windowing** selecciona un **subconjunto** del rango de HU y lo mapea a los 256 tonos de gris:

```
          Window Width (WW)
    ◄────────────────────────►
    ┌────────────────────────┐
    │    Rango visible       │
────┤  negro ◄──────► blanco ├────
    └────────────────────────┘
              ▲
         Window Center (WC)

Todo lo menor al borde izquierdo → negro
Todo lo mayor al borde derecho → blanco
```

Diferentes ventanas revelan diferentes estructuras:

| Ventana | Center (HU) | Width (HU) | ¿Qué se ve? |
|---------|-------------|------------|-------------|
| Pulmonar | -600 | 1500 | Parénquima pulmonar, vías aéreas |
| Mediastínica | 40 | 400 | Tejidos blandos, vasos, ganglios |
| Ósea | 400 | 1800 | Hueso, calcificaciones |
| Cerebral | 40 | 80 | Tejido cerebral (gris vs blanca) |

La fórmula del windowing es:

```
lower = center - width / 2
upper = center + width / 2
imagen_windowed = clip(imagen_hu, lower, upper)
imagen_normalizada = (imagen_windowed - lower) / (upper - lower)
```

In [ ]:
def apply_window(image_hu, center, width):
    """
    Aplica windowing a una imagen en unidades Hounsfield.

    Parámetros:
    - image_hu: array 2D con valores en HU
    - center: centro de la ventana (Window Center)
    - width: ancho de la ventana (Window Width)

    Retorna: array 2D normalizado entre 0 y 1

    Ejemplo: apply_window(image, center=40, width=400)
      → muestra solo el rango [-160, 240] HU
      → todo < -160 se ve negro, todo > 240 se ve blanco
    """
    lower = center - width / 2
    upper = center + width / 2

    # np.clip limita los valores al rango [lower, upper]
    windowed = np.clip(image_hu, lower, upper)

    # Normalizamos a [0, 1] para que matplotlib lo muestre correctamente
    normalized = (windowed - lower) / (upper - lower)

    return normalized

# Probamos con la ventana mediastínica
img_mediastinal = apply_window(image_hu, center=40, width=400)
print(f"Ventana mediastínica: centro=40, ancho=400")
print(f"Rango visible: [{40-400/2:.0f}, {40+400/2:.0f}] HU")
print(f"Resultado: valores entre {img_mediastinal.min():.2f} y {img_mediastinal.max():.2f}")

### 2.4 Ejercicio: Comparar ventanas

**Ejercicio 2.4:** Genera una figura con 4 subplots mostrando la misma imagen CT con las 4 ventanas estándar. Completa el código usando la función `apply_window()` que acabamos de definir.

Observa cómo la **misma imagen** revela estructuras completamente diferentes según la ventana elegida.

In [ ]:
# ── EJERCICIO 2.4: Comparar 4 ventanas ──

windows = {
    "Pulmonar\n(C=-600, W=1500)":     {"center": -600, "width": 1500},
    "Mediastínica\n(C=40, W=400)":     {"center": 40,   "width": 400},
    "Ósea\n(C=400, W=1800)":           {"center": 400,  "width": 1800},
    "Cerebral\n(C=40, W=80)":          {"center": 40,   "width": 80},
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (title, params) in zip(axes, windows.items()):
    # TU CÓDIGO AQUÍ: aplica la ventana y muestra la imagen
    # Usa: apply_window(image_hu, params["center"], params["width"])
    # Muestra con: ax.imshow(..., cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.suptitle("Misma imagen CT — 4 ventanas diferentes", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Pregunta 2.4:** ¿Qué estructuras anatómicas puedes distinguir en cada ventana? ¿Cuáles desaparecen? ¿Por qué un radiólogo necesita cambiar entre ventanas al leer un estudio?

*Tu respuesta aquí...*

### 2.5 Ventanas en MR: una comparación

La RM también usa windowing, pero a diferencia del CT, los valores de píxel en MR **no representan una escala física absoluta** como las unidades Hounsfield. El contraste en MR depende de los parámetros de adquisición (T1, T2, etc.).

Veamos cómo se ve el archivo MR grande (484x484) con sus dos ventanas embebidas en el DICOM.

In [ ]:
# Cargamos el MR grande (484x484) — tiene dos ventanas embebidas
mr_big = pydicom.dcmread(mr_big_path)
mr_pixels = mr_big.pixel_array.astype(np.float64)

# Este archivo SÍ tiene WindowCenter y WindowWidth (y son listas con 2 valores)
wc = mr_big.WindowCenter  # [450, 200]
ww = mr_big.WindowWidth   # [790, 443]

print(f"MR-SIEMENS: {mr_big.Rows}x{mr_big.Columns}")
print(f"Pixel range: {mr_pixels.min():.0f} a {mr_pixels.max():.0f}")
print(f"WindowCenter embebidos: {wc}")
print(f"WindowWidth embebidos:  {ww}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Imagen raw (sin windowing)
axes[0].imshow(mr_pixels, cmap='gray')
axes[0].set_title(f"MR raw\n(rango: {mr_pixels.min():.0f}-{mr_pixels.max():.0f})")
axes[0].axis('off')

# Ventana 1 (embebida en el DICOM)
center1, width1 = float(wc[0]), float(ww[0])
img_w1 = apply_window(mr_pixels, center1, width1)
axes[1].imshow(img_w1, cmap='gray')
axes[1].set_title(f"Ventana 1\n(C={center1:.0f}, W={width1:.0f})")
axes[1].axis('off')

# Ventana 2 (embebida en el DICOM)
center2, width2 = float(wc[1]), float(ww[1])
img_w2 = apply_window(mr_pixels, center2, width2)
axes[2].imshow(img_w2, cmap='gray')
axes[2].set_title(f"Ventana 2\n(C={center2:.0f}, W={width2:.0f})")
axes[2].axis('off')

plt.suptitle("MR con dos ventanas embebidas en el DICOM", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 2.6 Volumen 3D: múltiples cortes

Un estudio CT no es una sola imagen — es un **volumen tridimensional** compuesto por múltiples cortes transversales (axiales). Cada corte es un archivo DICOM individual, y juntos forman un volumen que se puede "rebanar" en tres planos:

```
         Coronal (de frente)
              ┌──────┐
             /      /│
            /      / │
           ┌──────┐  │   ← Axial (el plano nativo del CT)
           │      │  │
           │      │  /
           │      │ /    Sagital (de lado) →
           └──────┘
```

Usaremos la serie CT5N que tiene **5 cortes de 16x16 píxeles**. Es pequeña (datos de prueba), pero el proceso es exactamente igual al de un CT real con 100-500 cortes de 512x512.

El proceso es:
1. Cargar todos los archivos DICOM de la serie
2. Ordenarlos por posición espacial (`ImagePositionPatient`)
3. Apilarlos en un array 3D con numpy

In [ ]:
# ── Cargar y ordenar la serie CT ──
slices = []
for filepath in ct_series_files:
    ds = pydicom.dcmread(filepath)
    slices.append(ds)

# Ordenamos por la posición Z (tercer componente de ImagePositionPatient)
# Esto asegura que los cortes estén en orden espacial, de abajo a arriba
slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))

# Extraemos metadata espacial del primer corte
pixel_spacing = slices[0].PixelSpacing
slice_thickness = slices[0].SliceThickness

print(f"Serie cargada: {len(slices)} cortes")
print(f"Tamaño de cada corte: {slices[0].Rows} x {slices[0].Columns}")
print(f"PixelSpacing: {pixel_spacing} mm (distancia entre píxeles en X,Y)")
print(f"SliceThickness: {slice_thickness} mm (grosor de cada corte)")
print()

# Mostramos las posiciones Z de cada corte
for i, s in enumerate(slices):
    pos_z = float(s.ImagePositionPatient[2])
    print(f"  Corte {i}: posición Z = {pos_z:.1f} mm")

# Apilamos en un volumen 3D
# Convertimos cada corte a HU primero
slope = float(slices[0].RescaleSlope)
intercept = float(slices[0].RescaleIntercept)

volume = np.stack([
    s.pixel_array.astype(np.float64) * slope + intercept
    for s in slices
])

print(f"\nVolumen 3D: shape = {volume.shape}")
print(f"  {volume.shape[0]} cortes x {volume.shape[1]} filas x {volume.shape[2]} columnas")

### 2.7 Los tres planos anatómicos

Con el volumen 3D podemos visualizar tres planos de corte:

- **Axial** (horizontal): el plano nativo del CT. Se selecciona con `volume[z, :, :]`
- **Coronal** (frontal): de frente al paciente. Se selecciona con `volume[:, y, :]`
- **Sagital** (lateral): de lado. Se selecciona con `volume[:, :, x]`

> **Nota:** Esta serie de prueba tiene solo 5 cortes de 16x16 — los cortes coronal y sagital se verán como franjas estrechas (5 píxeles de alto). En un CT real de 500 cortes a 512x512, los tres planos se ven con resolución similar.

Para que los cortes no axiales se vean en **proporción correcta**, necesitamos ajustar el `aspect ratio` usando la información de PixelSpacing y SliceThickness del DICOM.

In [ ]:
# ── Visualizar los tres planos ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Calculamos aspect ratios correctos
ps_row = float(pixel_spacing[0])  # distancia entre filas (Y)
ps_col = float(pixel_spacing[1])  # distancia entre columnas (X)
st = float(slice_thickness)       # distancia entre cortes (Z)

# Plano axial: un corte horizontal (el corte del medio)
ax_idx = volume.shape[0] // 2
axes[0].imshow(volume[ax_idx, :, :], cmap='gray', aspect=ps_row/ps_col)
axes[0].set_title(f"Axial (corte {ax_idx})")
axes[0].set_xlabel("X")
axes[0].set_ylabel("Y")

# Plano coronal: un corte frontal (la fila del medio)
cor_idx = volume.shape[1] // 2
axes[1].imshow(volume[:, cor_idx, :], cmap='gray', aspect=st/ps_col)
axes[1].set_title(f"Coronal (fila {cor_idx})")
axes[1].set_xlabel("X")
axes[1].set_ylabel("Z (cortes)")

# Plano sagital: un corte lateral (la columna del medio)
sag_idx = volume.shape[2] // 2
axes[2].imshow(volume[:, :, sag_idx], cmap='gray', aspect=st/ps_row)
axes[2].set_title(f"Sagital (columna {sag_idx})")
axes[2].set_xlabel("Y")
axes[2].set_ylabel("Z (cortes)")

plt.suptitle(f"Tres planos anatómicos del volumen CT ({volume.shape[0]}x{volume.shape[1]}x{volume.shape[2]})",
             fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

print("Nota: los planos coronal y sagital se ven como franjas porque")
print(f"solo tenemos {volume.shape[0]} cortes. En un CT real de 500 cortes,")
print("los tres planos se verían con resolución comparable.")

**Pregunta 2.7:** ¿Por qué los cortes sagital y coronal se ven "estirados" si no ajustamos el aspect ratio? ¿Qué campos del DICOM se necesitan para calcular la proporción correcta?

*Tu respuesta aquí...*

**CHECKPOINT 2** — Haz commit de tu progreso:
```bash
git add -A && git commit -m "Checkpoint 2: windowing and 3D volume complete"
```

---

## Parte 3: Segmentación por Umbral

### 3.1 Histograma de intensidades

El histograma de valores HU nos muestra la **distribución de tejidos** en la imagen. Cada pico corresponde a un tipo de tejido con una densidad característica.

Esto es poderoso: sin saber nada de anatomía, podemos identificar qué tejidos están presentes en la imagen simplemente mirando la distribución de densidades.

In [ ]:
# ── Histograma de HU del CT ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma completo
axes[0].hist(image_hu.flatten(), bins=256, color='steelblue', edgecolor='none', alpha=0.7)
axes[0].set_xlabel('Unidades Hounsfield (HU)')
axes[0].set_ylabel('Frecuencia (número de píxeles)')
axes[0].set_title('Histograma completo de HU')

# Histograma anotado con rangos de tejido
axes[1].hist(image_hu.flatten(), bins=256, color='steelblue', edgecolor='none', alpha=0.7)
axes[1].set_xlabel('Unidades Hounsfield (HU)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Histograma con rangos de tejido identificados')

# Anotaciones de tejidos
tissue_ranges = [
    (-1000, -500, 'Aire', 'lightblue'),
    (-100,  -50,  'Grasa', 'yellow'),
    (0,     80,   'Tejido blando', 'salmon'),
    (300,   1200, 'Hueso', 'lightgray'),
]

for low, high, name, color in tissue_ranges:
    axes[1].axvspan(low, high, alpha=0.3, color=color, label=f'{name} ({low} a {high} HU)')

axes[1].legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

print("Observa los picos del histograma:")
print("  - El pico más grande probablemente corresponde al aire o fondo de la imagen")
print("  - Picos intermedios corresponden a tejidos blandos")
print("  - Valores altos corresponden a hueso")

### 3.2 Segmentación binaria

La segmentación más simple posible es el **umbral**: seleccionamos un rango de HU y marcamos todos los píxeles que caen dentro como "positivos" (1) y el resto como "negativos" (0). Esto crea una **máscara binaria**.

La función recibe los límites inferior y superior en HU y retorna una máscara booleana del mismo tamaño que la imagen.

In [ ]:
def segment_by_threshold(image_hu, low, high):
    """
    Crea una máscara binaria seleccionando píxeles en un rango de HU.

    Parámetros:
    - image_hu: array 2D con valores en HU
    - low: límite inferior del rango (inclusive)
    - high: límite superior del rango (inclusive)

    Retorna: array 2D booleano (True donde el píxel está en el rango)

    Ejemplo: segment_by_threshold(image, -100, 200)
      → True para tejido blando, False para todo lo demás
    """
    return (image_hu >= low) & (image_hu <= high)

# Ejemplo: segmentar hueso (HU > 300)
bone_mask = segment_by_threshold(image_hu, 300, 2000)
print(f"Píxeles clasificados como hueso: {bone_mask.sum()} de {bone_mask.size}")
print(f"Porcentaje: {bone_mask.sum() / bone_mask.size:.1%}")

### 3.3 Ejercicio: Segmentación de múltiples tejidos

**Ejercicio 3.3:** Crea máscaras de segmentación para tres tipos de tejido y visualízalas superpuestas sobre la imagen original.

Umbrales sugeridos:
- **Aire/fondo:** HU < -500
- **Tejido blando:** -100 a 200 HU
- **Hueso:** HU > 300

Completa el código para crear las máscaras y la visualización combinada.

In [ ]:
# ── EJERCICIO 3.3: Segmentación de múltiples tejidos ──

# Crea las tres máscaras
# TU CÓDIGO AQUÍ
# air_mask = segment_by_threshold(image_hu, ?, ?)
# soft_tissue_mask = segment_by_threshold(image_hu, ?, ?)
# bone_mask = segment_by_threshold(image_hu, ?, ?)

# Visualización: imagen original + máscaras superpuestas
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Imagen original en ventana mediastínica
img_display = apply_window(image_hu, 40, 400)
axes[0].imshow(img_display, cmap='gray')
axes[0].set_title('Original\n(ventana mediastínica)')
axes[0].axis('off')

# Máscara de aire
# TU CÓDIGO AQUÍ: muestra img_display en gris, superpón air_mask en azul con alpha=0.3
axes[1].set_title('Aire\n(< -500 HU)')
axes[1].axis('off')

# Máscara de tejido blando
# TU CÓDIGO AQUÍ
axes[2].set_title('Tejido blando\n(-100 a 200 HU)')
axes[2].axis('off')

# Máscara de hueso
# TU CÓDIGO AQUÍ
axes[3].set_title('Hueso\n(> 300 HU)')
axes[3].axis('off')

plt.suptitle('Segmentación por umbral de HU', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Mapa de tejidos combinado

Combinamos las tres máscaras en una sola imagen donde cada tejido tiene un color diferente. Esto es una forma rudimentaria de "atlas anatómico automático".

In [ ]:
# ── Mapa combinado de tejidos ──
# Creamos una imagen RGB donde cada canal representa un tejido
tissue_map = np.zeros((*image_hu.shape, 3))  # 3 canales: R, G, B

# Las máscaras se deben crear arriba. Si no completaste el ejercicio,
# descomenta estas líneas:
# air_mask = segment_by_threshold(image_hu, -1000, -500)
# soft_tissue_mask = segment_by_threshold(image_hu, -100, 200)
# bone_mask = segment_by_threshold(image_hu, 300, 2000)

try:
    tissue_map[air_mask, 2] = 1.0          # Azul para aire
    tissue_map[soft_tissue_mask, 0] = 1.0  # Rojo para tejido blando
    tissue_map[bone_mask, 1] = 1.0         # Verde para hueso (visible en imagen)
    tissue_map[bone_mask, 2] = 0.5         # Cian para hueso

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(apply_window(image_hu, 40, 400), cmap='gray')
    axes[0].set_title('Imagen original')
    axes[0].axis('off')

    axes[1].imshow(tissue_map)
    axes[1].set_title('Mapa de tejidos\n(Azul=aire, Rojo=tejido blando, Cian=hueso)')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    # Estadísticas
    total = image_hu.size
    print(f"=== Distribución de tejidos ===")
    print(f"  Aire:           {air_mask.sum():5d} píxeles ({air_mask.sum()/total:.1%})")
    print(f"  Tejido blando:  {soft_tissue_mask.sum():5d} píxeles ({soft_tissue_mask.sum()/total:.1%})")
    print(f"  Hueso:          {bone_mask.sum():5d} píxeles ({bone_mask.sum()/total:.1%})")
    print(f"  No clasificado: {total - air_mask.sum() - soft_tissue_mask.sum() - bone_mask.sum():5d} píxeles")
except NameError:
    print("Completa el ejercicio 3.3 primero para crear las máscaras.")

**CHECKPOINT 3** — Haz commit de tu progreso:
```bash
git add -A && git commit -m "Checkpoint 3: segmentation complete"
```

### Pregunta de reflexión — Segmentación

**Pregunta 3.1:** La segmentación por umbral funciona razonablemente bien en CT pero es mucho menos efectiva en MRI. ¿Por qué? (Pista: piensa en qué representan los valores de píxel en cada modalidad. En CT, los HU miden una propiedad física absoluta — ¿qué pasa en MRI?)

*Tu respuesta aquí...*

---

## Parte 4: Reflexión Final

Responde las siguientes preguntas en las celdas de abajo.

**Pregunta 4.1:** Imagina que recibes un dataset de 10,000 archivos DICOM de diferentes hospitales para entrenar un modelo de machine learning. ¿Qué pasos de preprocesamiento necesitarías hacer antes de poder usarlos? Menciona al menos 3 consideraciones (piensa en: privacidad, estandarización, calidad).

*Tu respuesta aquí...*

**Pregunta 4.2:** En la clase teórica se presentó el framework **Dato Crudo → Representación → Análisis**. ¿Cómo se aplica este framework a lo que hicimos en este lab? Identifica qué fue el dato crudo, qué representaciones creamos, y qué análisis (aunque básico) realizamos.

*Tu respuesta aquí...*

**Pregunta 4.3:** Si tuvieras que explicarle a un médico que nunca ha programado por qué es importante trabajar con archivos DICOM en vez de imágenes PNG/JPEG, ¿qué le dirías? Resume en 3-4 oraciones.

*Tu respuesta aquí...*

**CHECKPOINT 4 (final)** — Haz commit de tu trabajo completo:
```bash
git add -A && git commit -m "Checkpoint 4: lab complete with reflections"
```

---

## Resumen

En este lab exploramos el estándar DICOM y aprendimos que las imágenes médicas son mucho más que matrices de píxeles:

1. **Metadata clínica:** Un archivo DICOM contiene información del paciente, del estudio y del equipo — todo empaquetado junto con la imagen.
2. **Unidades Hounsfield:** Los valores crudos de píxel se convierten a una escala física estandarizada usando RescaleSlope e Intercept.
3. **Windowing:** La misma imagen CT revela estructuras completamente diferentes según la ventana de visualización elegida — un concepto fundamental en radiología.
4. **Volumen 3D:** Un estudio CT es un volumen tridimensional que se puede visualizar en múltiples planos anatómicos.
5. **Segmentación:** Los umbrales de HU permiten separar tejidos automáticamente, aunque esta técnica es limitada y solo funciona bien en CT.

### Conexión con el curso

- **Lab 2 (Terminología):** Así como los diagnósticos necesitan códigos estandarizados (ICD-10, SNOMED), las imágenes necesitan un formato estandarizado (DICOM) para ser interoperables.
- **Lab 4 (Modelado):** La ingeniería de características que hicimos con datos tabulares (extraer features de CSVs) tiene su equivalente en imágenes: extraer features de píxeles (histogramas, segmentaciones, texturas).
- **Próximo paso:** Los modelos de deep learning (CNN) automatizan la extracción de features de imágenes, reemplazando la segmentación manual por umbral con aprendizaje de patrones complejos.

### Referencia

- Documentación de pydicom: https://pydicom.github.io/pydicom/stable/
- Tabla de unidades Hounsfield: https://radiopaedia.org/articles/hounsfield-unit
- DICOM Standard Browser: https://dicom.innolitics.com/ciods